In [1]:
import cv2
import torch
import timm
import numpy as np
from torchvision import transforms
from tqdm import tqdm
from pathlib import Path

# ------------------ CONFIG ------------------
VIDEO_PATH = "D:/Projects/Major Project/Deepfake Detection/datasets/dfd/manipulated/01_02__exit_phone_room__YVGY8LOK.mp4"
MODEL_PATH = "D:/Projects/Major Project/Deepfake Detection/models/vit_deit_tiny_5epochs.pth"
FRAME_SKIP = 5        # process every 5th frame
MAX_FRAMES = 100      # limit frames per video
IMG_SIZE = 224
CLASS_NAMES = ["fake", "real"]

# ------------------ DEVICE ------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")


🚀 Using device: cpu


In [2]:

# ------------------ TRANSFORMS ------------------
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------ LOAD MODEL ------------------
model = timm.create_model(
    'deit_tiny_patch16_224',
    pretrained=False,
    num_classes=2
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=192, out_features=576, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=192, out_features=192, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=192, out_features=768, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Identity()


In [3]:

# ------------------ VIDEO PROCESSING ------------------
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("❌ Cannot open video")

frame_count = 0
processed = 0

frame_probs = []   # fake probability
frame_preds = []

print("\n🎬 Processing video frames...\n")

with torch.no_grad():
    for _ in tqdm(range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))), desc="Frames"):
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % FRAME_SKIP != 0:
            frame_count += 1
            continue

        frame_count += 1
        processed += 1

        # BGR → RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Transform
        input_tensor = transform(frame_rgb).unsqueeze(0).to(device)

        # ViT inference
        outputs = model(input_tensor)
        probs = torch.softmax(outputs, dim=1)

        fake_prob = probs[0][0].item()
        pred = torch.argmax(probs, dim=1).item()

        frame_probs.append(fake_prob)
        frame_preds.append(pred)

        if processed >= MAX_FRAMES:
            break

cap.release()



🎬 Processing video frames...



Frames: 100%|██████████| 210/210 [00:05<00:00, 37.53it/s]


In [4]:

# ------------------ AGGREGATION ------------------
frame_probs = np.array(frame_probs)
frame_preds = np.array(frame_preds)

mean_fake = frame_probs.mean()
max_fake = frame_probs.max()
vote_fake = np.mean(frame_preds == 0)

final_score = 0.6 * mean_fake + 0.4 * max_fake
final_label = "FAKE" if final_score > 0.5 else "REAL"

# ------------------ RESULTS ------------------
print("\n" + "="*60)
print("🎯 VIDEO-LEVEL ViT EVALUATION RESULT")
print("="*60)

print(f"📁 Video: {Path(VIDEO_PATH).name}")
print(f"🖼️  Frames analyzed: {len(frame_probs)}")
print(f"\n📊 Frame-level stats:")
print(f"   Mean fake probability : {mean_fake:.4f}")
print(f"   Max fake probability  : {max_fake:.4f}")
print(f"   Fake vote ratio       : {vote_fake:.2%}")

print(f"\n🏁 FINAL DECISION:")
print(f"   ▶ Label      : {final_label}")
print(f"   ▶ Confidence : {final_score:.4f}")

print("="*60)



🎯 VIDEO-LEVEL ViT EVALUATION RESULT
📁 Video: 01_02__exit_phone_room__YVGY8LOK.mp4
🖼️  Frames analyzed: 42

📊 Frame-level stats:
   Mean fake probability : 0.0136
   Max fake probability  : 0.0306
   Fake vote ratio       : 0.00%

🏁 FINAL DECISION:
   ▶ Label      : REAL
   ▶ Confidence : 0.0204
